# kev-8b inference (RQ6)

Colab (L4) record of the kev-8b run. `kev.serve` from `github.com/jaredpalmer/kev` runs in its own `uv` venv, and `src.judge_kev` calls it over HTTP (D27). Clean and verbose, both orders, one call each; calls over kev's 8,160-token serving ceiling are skipped. Output: `runs/kev_8b/kev.jsonl`, copied back locally (D17).

**Step 1 — mount Drive:**

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


**Step 2 — pull the latest project code**

In [2]:
import getpass
token = getpass.getpass('Enter your GitHub PAT: ')
!cd /content/drive/MyDrive/judge-calibration && git pull https://{token}@github.com/senguptashubham/judge-calibration.git main


Enter your GitHub PAT: ··········
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 3.24 KiB | 1024 bytes/s, done.
From https://github.com/senguptashubham/judge-calibration
 * branch            main       -> FETCH_HEAD
Updating 6f8a2a6..775cc94
Fast-forward
 DECISIONS.md     |  6 ++++++
 TASKS.md         |  1 +
 src/judge_kev.py | 35 ++++++++++++++++++++---------------
 3 files changed, 27 insertions(+), 15 deletions(-)


gets you `judge_kev.py`/`run_kev.yaml`

**Step 3 — clean, fresh clone + venv for kev, entirely in ephemeral** `/content` (not Drive):

In [3]:
%cd /content
!rm -rf kev
!git clone https://github.com/jaredpalmer/kev.git
%cd /content/kev
!pip install -q uv
!uv sync --extra serve
!uv pip install --python /content/kev/.venv/bin/python pandas pyarrow pyyaml requests


/content
Cloning into 'kev'...
remote: Enumerating objects: 3155, done.
remote: Counting objects: 100% (733/733), done.
remote: Compressing objects: 100% (228/228), done.
remote: Total 3155 (delta 548), reused 525 (delta 490), pack-reused 2422 (from 2)
Receiving objects: 100% (3155/3155), 27.41 MiB | 20.01 MiB/s, done.
Resolving deltas: 100% (1995/1995), done.
/content/kev
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 122.9 MB/s eta 0:00:00
Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 120 packages in 1ms
Prepared 113 packages in 28.02s
Installed 113 packages in 2.01s
 + accelerate==1.15.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + attrs==26.1.0
 + cbor2==6.1.4
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + contourpy==1.4.0
 + cycler==0.12.1
 + datasets==5.0.1
 + dill==0.4.1
 + fasta

That last line adds the project's own light dependencies (pandas/pyarrow/pyyaml/requests) into kev's venv, alongside the transformers/torch/peft it already installs — so one Python interpreter has everything judge_kev.py needs.

**Step 4 — launch** `kev.serve`, then confirm it's actually up before touching anything else:

In [4]:
!KEV_DTYPE=bf16 KEV_MERGE=0 KEV_ATTN=sdpa setsid nohup uv run --extra serve python -m kev.serve \
    --run jaredpalmer/kev-8b --port 8008 > /content/kev/kev_serve.log 2>&1 < /dev/null &


Repeat next cell (non-blocking, safe to re-run) until you see `Uvicorn running on http://127.0.0.1:8008`.

In [10]:
!tail -n 100 /content/kev/kev_serve.log


Loading weights: 100%|██████████| 399/399 [00:00<00:00, 7853.54it/s]
INFO:     Started server process [4597]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8008 (Press CTRL+C to quit)


Since this is a fresh runtime, `nvidia-smi` should already be clean, but a quick check costs nothing:

In [11]:
!nvidia-smi


Tue Sep 22 18:29:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P0             27W /   72W |   15138MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

healthy looks like: one process, 15,138 MiB

If any process running that you want to kill, use the below cell

In [ ]:
!kill -9 {process_id}
!nvidia-smi

**Step 5 — smoke test, from the Drive-mounted project directory:**

In [12]:
%cd /content/drive/MyDrive/judge-calibration
!PYTHONPATH=/content/drive/MyDrive/judge-calibration /content/kev/.venv/bin/python -m src.judge_kev \
    --config configs/run_kev.yaml --n-items 10


/content/drive/MyDrive/judge-calibration









Then check it:

In [13]:
!cat runs/kev_8b/kev.jsonl


{"item_id": "8a43171bb8b85aa8", "question_id": 95, "category": null, "model_a": "gpt-3.5-turbo", "model_b": "claude-v1", "turn": 2, "condition": "clean", "order": "AB", "judge_model": "jaredpalmer/kev-8b", "git_sha": "775cc94f666a8b2f3c97f8b80e3dbfdff4b96ddd", "state_tokens": 431, "skipped": false, "ok": true, "http_status": 200, "raw_response": {"model": "kev-latest", "answers": {"verdict": {"type": "choice", "choice": "A", "confidence": 0.6328, "probabilities": {"A": 0.8164, "B": 0.1836}}}, "usage": {"input_tokens": 451, "output_tokens": 53}, "latency_ms": 641.4}, "error": null}
{"item_id": "8a43171bb8b85aa8", "question_id": 95, "category": null, "model_a": "gpt-3.5-turbo", "model_b": "claude-v1", "turn": 2, "condition": "clean", "order": "BA", "judge_model": "jaredpalmer/kev-8b", "git_sha": "775cc94f666a8b2f3c97f8b80e3dbfdff4b96ddd", "state_tokens": 431, "skipped": false, "ok": true, "http_status": 200, "raw_response": {"model": "kev-latest", "answers": {"verdict": {"type": "choice"

Expect 40 rows (10 items × 4 calls). Look for `ok: true` with a `choice`/`probabilities`, and if anything's `skipped: true`, the reason should be `over_max_state_tokens`, nothing else.

**Step 6 — if that looks right, the full run** (same command, drop `--n-items`)

In [ ]:
%cd /content/drive/MyDrive/judge-calibration
!PYTHONPATH=/content/drive/MyDrive/judge-calibration /content/kev/.venv/bin/python -m src.judge_kev \
    --config configs/run_kev.yaml


/content/drive/MyDrive/judge-calibration


It's resumable — a runtime death just means re-running the same command picks up where it left off.

**Step 7 — results land in** `runs/kev_8b/kev.jsonl`